Objectives (bi-objective): **time penalty** + **reverse penalty** (`reward[0]` and `reward[1]` from the default 3D vector).

Key change:
- Use a PyTorch **DQN** inner solver (linear scalarization of the two objectives, KL-regularized toward the cached center policy).
- Baselines include equal-spaced weights, CDF refinement, souping, **OLS (Linear Support)**, **UMOD** (DST-style outer loop), and metrics aggregated by **mean/std across seeds**.


In [10]:
import os
import math
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

import mo_gymnasium as mo_gym

from scipy.interpolate import PchipInterpolator

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cpu


## A. Environment + inner solver

In [ ]:
# MO-MountainCar: continuous state (position, velocity), discrete actions {left, coast, right}

# Match FishWood: scalarized objective minus tau * KL toward reference policy (uniform over actions).
TAU = 0.05
ACTION_DIM = 3

# # Easier starts: sample initial position from a range closer to the goal.
# # Increase these toward 0.5 to make starts even easier.
# EASY_START_LOW = 0.40
# EASY_START_HIGH = 0.45
# USE_EASY_START = False


def set_all_seeds(seed: int):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def make_env(env_name="mo-mountaincar-v0", max_episode_steps=200):
    """Default MO-MountainCar with explicit TimeLimit=200 steps."""
    return mo_gym.make(env_name, max_episode_steps=int(max_episode_steps))


def reset_env(env, seed=None):
    if seed is None:
        out = env.reset()
    else:
        try:
            out = env.reset(seed=int(seed))
        except TypeError:
            env.seed(int(seed))
            out = env.reset()

    state = out[0] if isinstance(out, tuple) else out

    # Optional easier curriculum: shift initial position closer to the goal.
    if USE_EASY_START:
        easy_pos = float(np.random.uniform(EASY_START_LOW, EASY_START_HIGH))
        easy_vel = 0.0
        easy_state = np.array([easy_pos, easy_vel], dtype=np.float32)

        try:
            env.unwrapped.state = easy_state.copy()
            state = easy_state
        except Exception:
            # If env backend does not expose state, keep original reset state.
            pass

    return state


def step_env(env, action):
    out = env.step(int(action))
    if len(out) == 5:
        obs, reward, terminated, truncated, info = out
        done = terminated or truncated
        return obs, reward, bool(done), info, bool(terminated), bool(truncated)
    obs, reward, done, info = out
    terminated = bool(done)
    truncated = bool(info.get("TimeLimit.truncated", False)) if isinstance(info, dict) else False
    return obs, reward, bool(done), info, terminated, truncated


class PolicyNet(nn.Module):
    def __init__(self, state_dim=2, action_dim=3, hidden=64):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden)
        self.fc2 = nn.Linear(hidden, action_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)  # logits


@torch.no_grad()
def flatten_params(model):
    return torch.cat([p.detach().flatten() for p in model.parameters()]).cpu().numpy().astype(np.float32)


@torch.no_grad()
def set_params_from_vector(model, vec):
    vec_t = torch.tensor(vec, dtype=torch.float32, device=next(model.parameters()).device)
    idx = 0
    for p in model.parameters():
        n = p.numel()
        p.copy_(vec_t[idx:idx+n].view_as(p))
        idx += n


@torch.no_grad()
def softmax_action_probs(model, state_np):
    s = torch.tensor(state_np, dtype=torch.float32, device=next(model.parameters()).device).unsqueeze(0)
    logits = model(s)
    probs = torch.softmax(logits, dim=-1).squeeze(0)
    return probs


def vector_reward_from_step(env_reward):
    """Bi-objective step reward: [time penalty, reverse penalty]. Forward penalty from env is ignored."""
    r = np.asarray(env_reward, dtype=np.float32).reshape(-1)
    if r.size >= 2:
        return np.array([r[0], r[1]], dtype=np.float32)
    if r.size == 1:
        return np.array([r[0], 0.0], dtype=np.float32)
    return np.zeros(2, dtype=np.float32)


def make_uniform_pi_ref_logits(action_dim=ACTION_DIM, device=DEVICE):
    """Tabular FishWood uses pi_ref(s,a)=1/A; for logits, all zeros give softmax = uniform."""
    return torch.zeros(int(action_dim), dtype=torch.float32, device=device)


def kl_categorical_logits_1d(logits_p, logits_q):
    """KL( softmax(logits_p) || softmax(logits_q) ), both length A."""
    logp = F.log_softmax(logits_p, dim=-1)
    logq = F.log_softmax(logits_q, dim=-1)
    p = logp.exp()
    return torch.sum(p * (logp - logq))


def discounted_returns(rews, gamma):
    out = []
    g = 0.0
    for r in reversed(rews):
        g = r + gamma * g
        out.append(g)
    out.reverse()
    return np.asarray(out, dtype=np.float32)


def train_policy_scalarization(
    w,
    seed,
    env_name="mo-mountaincar-v0",
    gamma=0.995,
    tau=TAU,
    lr=1e-3,
    train_episodes=200,
    max_steps=200,
    entropy_coef=1e-3,
    init_params=None,
    hidden=64,
    device=DEVICE,
    eval_greedy=True,
):
    """REINFORCE inner solver for scalarized two-objective reward."""
    set_all_seeds(seed)
    env = make_env(env_name)

    model = PolicyNet(hidden=hidden).to(device)
    if init_params is not None:
        set_params_from_vector(model, init_params)

    optim = torch.optim.Adam(model.parameters(), lr=lr)
    loss_history = []
    scalar_obj_history = []

    for ep in range(int(train_episodes)):
        state = reset_env(env, seed=seed * 100_003 + ep)
        log_probs = []
        entropies = []
        scalar_rewards = []

        for _ in range(int(max_steps)):
            s = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            logits = model(s)
            dist = torch.distributions.Categorical(logits=logits)
            action = dist.sample()

            next_state, env_reward, done, info, terminated, truncated = step_env(env, action.item())
            rv = vector_reward_from_step(env_reward)
            scalar = (1.0 - float(w)) * rv[0] + float(w) * rv[1]

            # print(f"scalar={scalar}, rv={rv}")
            log_probs.append(dist.log_prob(action).squeeze(0))
            entropies.append(dist.entropy().squeeze(0))

            state = next_state
            # if terminated:
            #     scalar += 100
            scalar_rewards.append(float(scalar))
            if done:
                break

        G = discounted_returns(scalar_rewards, gamma)
        scalar_obj_history.append(float(G[0]) if len(G) > 0 else 0.0)
        G = torch.tensor(G, dtype=torch.float32, device=device)
        # if len(G) > 1:
        #     G = (G - G.mean()) / (G.std() + 1e-8)

        policy_loss = []
        for t in range(len(log_probs)):
            policy_loss.append(-log_probs[t] * G[t])
        loss = torch.stack(policy_loss).sum()

        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optim.step()

        loss_history.append(float(loss.detach().cpu().item()))

    out = evaluate_policy_multiobjective(
        model,
        env_name=env_name,
        n_eval_episodes=16,
        max_steps=max_steps,
        gamma=gamma,
        seed=seed + 7_777,
        device=device,
        greedy=eval_greedy,
    )
    out["w"] = float(w)
    out["params"] = flatten_params(model)
    out["loss_history"] = np.asarray(loss_history, dtype=np.float32)
    out["scalar_obj_history"] = np.asarray(scalar_obj_history, dtype=np.float32)
    print(f"[train] w={float(w):.3f}, epochs_trained={len(loss_history)}")
    env.close()
    return out


@torch.no_grad()
def evaluate_policy_multiobjective(
    model,
    env_name="mo-mountaincar-v0",
    n_eval_episodes=16,
    max_steps=200,
    gamma=0.995,
    seed=0,
    device=DEVICE,
    greedy=False,
):
    env = make_env(env_name)
    returns = []
    ep_lengths = []
    ep_success = []
    ep_truncated = []

    for ep in range(int(n_eval_episodes)):
        state = reset_env(env, seed=seed * 9_973 + ep)
        g = np.zeros(2, dtype=np.float64)
        disc = 1.0
        length = 0
        success = False
        trunc = False

        for _ in range(int(max_steps)):
            probs = softmax_action_probs(model, state)
            if greedy:
                action = int(torch.argmax(probs).item())
            else:
                action = int(torch.distributions.Categorical(probs=probs).sample().item())

            state, env_reward, done, info, terminated, truncated = step_env(env, action)
            rv = vector_reward_from_step(env_reward)
            g += disc * rv
            disc *= gamma
            length += 1

            if done:
                success = bool(terminated)
                trunc = bool(truncated)
                break

        returns.append(g)
        ep_lengths.append(length)
        ep_success.append(1.0 if success else 0.0)
        ep_truncated.append(1.0 if trunc else 0.0)

    env.close()
    arr = np.asarray(returns, dtype=np.float64)
    mean_vec = np.mean(arr, axis=0)
    return {
        "R1": float(mean_vec[0]),
        "R2": float(mean_vec[1]),
        "all_returns": arr,
        "avg_ep_len": float(np.mean(ep_lengths)) if ep_lengths else np.nan,
        "success_rate": float(np.mean(ep_success)) if ep_success else np.nan,
        "trunc_rate": float(np.mean(ep_truncated)) if ep_truncated else np.nan,
        "eval_greedy": bool(greedy),
    }


In [12]:
# Sanity check: Discrete(3) actions; env gives 3D reward; we use only [time, reverse]
env_chk = make_env()
obs = reset_env(env_chk, seed=123)
obs2, rew, done, info, terminated, truncated = step_env(env_chk, 1)  # action 1 = don't accelerate
print("Action space:", env_chk.action_space)
print("Sample reward:", rew, "| shape:", np.asarray(rew).shape)
r = np.asarray(rew).reshape(-1)
assert getattr(env_chk.action_space, "n", None) == 3, "Expected Discrete(3) action space"
assert r.size == 3, f"Expected 3D reward [time, reverse, forward], got reward {rew}"
rv = vector_reward_from_step(rew)
print("Vector [time, reverse] (forward ignored):", rv)
print("R2 (reverse penalty):", float(rv[1]))
env_chk.close()

Action space: Discrete(3)
Sample reward: [-1.  0.  0.] | shape: (3,)
Vector [time, reverse] (forward ignored): [-1.  0.]
R2 (reverse penalty): 0.0


/opt/anaconda3/envs/course/lib/python3.13/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


## B. Comparison methods (same structure as FishWood)

In [ ]:
class QNet(nn.Module):
    def __init__(self, state_dim=2, action_dim=3, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )

    def forward(self, x):
        return self.net(x)


@torch.no_grad()
def evaluate_q_policy_multiobjective(
    qnet,
    env_name="mo-mountaincar-v0",
    n_eval_episodes=16,
    max_steps=200,
    gamma=0.995,
    tau=TAU,
    ref_params=None,
    seed=0,
    device=DEVICE,
    greedy=True,
):
    env = make_env(env_name)
    returns = []
    ep_lengths = []
    ep_success = []
    ep_truncated = []
    kl_returns = []

    qnet.eval()

    qref = None
    if ref_params is not None:
        qref = QNet(hidden=128).to(device)
        set_params_from_vector(qref, np.asarray(ref_params, dtype=np.float32))
        qref.eval()

    for ep in range(int(n_eval_episodes)):
        state = reset_env(env, seed=seed * 9_973 + ep)
        g = np.zeros(2, dtype=np.float64)
        kl_ret = 0.0
        disc = 1.0
        length = 0
        success = False
        trunc = False

        for _ in range(int(max_steps)):
            s = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            q = qnet(s).squeeze(0)
            if greedy:
                action = int(torch.argmax(q).item())
            else:
                action = int(torch.distributions.Categorical(logits=q).sample().item())

            pi = F.softmax(q, dim=0)
            log_pi = F.log_softmax(q, dim=0)
            if qref is None:
                log_ref = -math.log(float(ACTION_DIM))
                kl_to_ref = torch.sum(pi * (log_pi - log_ref)).item()
            else:
                with torch.no_grad():
                    q_ref = qref(s).squeeze(0)
                    log_pi_ref = F.log_softmax(q_ref, dim=0)
                kl_to_ref = torch.sum(pi * (log_pi - log_pi_ref)).item()

            state, env_reward, done, info, terminated, truncated = step_env(env, action)
            rv = vector_reward_from_step(env_reward)
            reg = float(tau) * float(kl_to_ref)
            g += disc * (rv - reg)
            kl_ret += disc * float(kl_to_ref)
            disc *= gamma
            length += 1

            if done:
                success = bool(terminated)
                trunc = bool(truncated)
                break

        returns.append(g)
        kl_returns.append(float(kl_ret))
        ep_lengths.append(length)
        ep_success.append(1.0 if success else 0.0)
        ep_truncated.append(1.0 if trunc else 0.0)

    env.close()
    arr = np.asarray(returns, dtype=np.float64)
    mean_vec = np.mean(arr, axis=0)
    mean_kl = float(np.mean(kl_returns)) if kl_returns else 0.0
    reg = float(tau) * mean_kl
    return {
        "R1": float(mean_vec[0]),
        "R2": float(mean_vec[1]),
        "R1_raw": float(mean_vec[0] + reg),
        "R2_raw": float(mean_vec[1] + reg),
        "KL": mean_kl,
        "reg_term": reg,
        "all_returns": arr,
        "avg_ep_len": float(np.mean(ep_lengths)) if ep_lengths else np.nan,
        "success_rate": float(np.mean(ep_success)) if ep_success else np.nan,
        "trunc_rate": float(np.mean(ep_truncated)) if ep_truncated else np.nan,
        "eval_greedy": bool(greedy),
    }


def train_dqn_scalarization(
    w,
    seed,
    env_name="mo-mountaincar-v0",
    gamma=0.995,
    tau=TAU,
    lr=1e-3,
    train_episodes=200,
    init_params=None,
    ref_params=None,
    max_steps=200,
    batch_size=128,
    replay_capacity=20000,
    warmup_steps=1000,
    target_update_every=200,
    ema_tau=0.01,
    use_ema_target=True,
    eval_use_ema=True,
    eps_start=0.95,
    eps_end=0.05,
    eps_decay=5000.0,
    hidden=128,
    device=DEVICE,
    eval_greedy=True,
):
    set_all_seeds(seed)
    env = make_env(env_name)

    qnet = QNet(hidden=hidden).to(device)
    qtarget = QNet(hidden=hidden).to(device)
    if init_params is not None:
        set_params_from_vector(qnet, init_params)
    qtarget.load_state_dict(qnet.state_dict())
    qtarget.eval()

    qref = None
    if ref_params is not None:
        qref = QNet(hidden=hidden).to(device)
        set_params_from_vector(qref, np.asarray(ref_params, dtype=np.float32))
        qref.eval()

    optim = torch.optim.Adam(qnet.parameters(), lr=lr)

    states = np.zeros((int(replay_capacity), 2), dtype=np.float32)
    actions = np.zeros((int(replay_capacity),), dtype=np.int64)
    rewards = np.zeros((int(replay_capacity),), dtype=np.float32)
    next_states = np.zeros((int(replay_capacity), 2), dtype=np.float32)
    dones = np.zeros((int(replay_capacity),), dtype=np.float32)

    mem_size = 0
    ptr = 0
    total_steps = 0
    loss_history = []
    scalar_obj_history = []
    neg_kl_tau_history = []

    for ep in range(int(train_episodes)):
        state = reset_env(env, seed=seed * 100_003 + ep)
        ep_scalar = 0.0
        ep_neg_kl_tau = 0.0
        disc = 1.0

        for _ in range(int(max_steps)):
            total_steps += 1
            eps = float(eps_end + (eps_start - eps_end) * np.exp(-total_steps / eps_decay))

            if np.random.rand() < eps:
                action = np.random.randint(ACTION_DIM)
            else:
                s = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
                with torch.no_grad():
                    action = int(torch.argmax(qnet(s), dim=1).item())

            s_online = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            q_online = qnet(s_online).squeeze(0)
            log_pi_online = F.log_softmax(q_online, dim=0)
            pi_online = torch.exp(log_pi_online)
            if qref is None:
                log_ref_online = -math.log(float(ACTION_DIM))
                kl_to_ref_online = torch.sum(pi_online * (log_pi_online - log_ref_online)).item()
            else:
                with torch.no_grad():
                    q_ref_online = qref(s_online).squeeze(0)
                    log_pi_ref_online = F.log_softmax(q_ref_online, dim=0)
                kl_to_ref_online = torch.sum(pi_online * (log_pi_online - log_pi_ref_online)).item()

            next_state, env_reward, done, _, _, _ = step_env(env, action)
            rv = vector_reward_from_step(env_reward)
            neg_kl_tau_step = -float(tau) * float(kl_to_ref_online)
            f1 = float(rv[0]) + neg_kl_tau_step
            f2 = float(rv[1]) + neg_kl_tau_step
            scalar = (1.0 - float(w)) * f1 + float(w) * f2

            states[ptr] = np.asarray(state, dtype=np.float32)
            actions[ptr] = int(action)
            rewards[ptr] = float(scalar)
            next_states[ptr] = np.asarray(next_state, dtype=np.float32)
            dones[ptr] = 1.0 if done else 0.0

            ptr = (ptr + 1) % int(replay_capacity)
            mem_size = min(mem_size + 1, int(replay_capacity))

            ep_scalar += disc * float(scalar)
            ep_neg_kl_tau += disc * float(neg_kl_tau_step)
            disc *= gamma
            state = next_state

            if mem_size >= max(int(batch_size), int(warmup_steps)):
                idx = np.random.randint(0, mem_size, size=int(batch_size))

                b_s = torch.tensor(states[idx], dtype=torch.float32, device=device)
                b_a = torch.tensor(actions[idx], dtype=torch.int64, device=device).unsqueeze(1)
                b_r = torch.tensor(rewards[idx], dtype=torch.float32, device=device).unsqueeze(1)
                b_ns = torch.tensor(next_states[idx], dtype=torch.float32, device=device)
                b_d = torch.tensor(dones[idx], dtype=torch.float32, device=device).unsqueeze(1)

                q_all = qnet(b_s)
                q = q_all.gather(1, b_a)
                with torch.no_grad():
                    q_next = qtarget(b_ns).max(dim=1, keepdim=True)[0]
                    target = b_r + float(gamma) * (1.0 - b_d) * q_next

                td_loss = F.mse_loss(q, target)
                loss = td_loss
                optim.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(qnet.parameters(), 5.0)
                optim.step()
                loss_history.append(float(loss.detach().cpu().item()))

                if use_ema_target:
                    # Exponential moving average target update:
                    # theta_target <- ema_tau * theta_online + (1-ema_tau) * theta_target
                    with torch.no_grad():
                        for p_tgt, p_online in zip(qtarget.parameters(), qnet.parameters()):
                            p_tgt.data.mul_(1.0 - float(ema_tau)).add_(float(ema_tau) * p_online.data)
                elif total_steps % int(target_update_every) == 0:
                    qtarget.load_state_dict(qnet.state_dict())

            if done:
                break

        scalar_obj_history.append(float(ep_scalar))
        neg_kl_tau_history.append(float(ep_neg_kl_tau))

    eval_net = qtarget if eval_use_ema else qnet
    out = evaluate_q_policy_multiobjective(
        eval_net,
        env_name=env_name,
        n_eval_episodes=16,
        max_steps=max_steps,
        gamma=gamma,
        tau=tau,
        ref_params=ref_params,
        seed=seed + 9_200,
        device=device,
        greedy=eval_greedy,
    )
    out["w"] = float(w)
    out["params"] = flatten_params(qnet)
    out["loss_history"] = np.asarray(loss_history, dtype=np.float32)
    out["scalar_obj_history"] = np.asarray(scalar_obj_history, dtype=np.float32)
    out["neg_kl_tau_history"] = np.asarray(neg_kl_tau_history, dtype=np.float32)
    out["ema_tau"] = float(ema_tau)
    out["use_ema_target"] = bool(use_ema_target)
    out["eval_use_ema"] = bool(eval_use_ema)
    env.close()
    return out


def run_equispace_weights_dqn(
    N_points,
    seed,
    train_episodes=200,
    lr=1e-3,
    gamma=0.995,
    tau=TAU,
    max_steps=200,
    eval_greedy=True,
):
    weights = np.linspace(0.0, 1.0, int(N_points))
    results = []
    pf = []
    for i, w in enumerate(weights):
        sol = train_dqn_scalarization(
            w=float(w),
            seed=int(seed + 40_009 * i),
            train_episodes=train_episodes,
            lr=lr,
            gamma=gamma,
            tau=tau,
            max_steps=max_steps,
            eval_greedy=eval_greedy,
        )
        results.append(sol)
        pf.append([sol["R1"], sol["R2"]])

    return {
        "weights": np.asarray(weights, dtype=np.float64),
        "results": results,
        "pf": np.asarray(pf, dtype=np.float64),
    }


def get_or_train_center_dqn_solution(
    seed,
    center_w=0.6,
    center_train_episodes=2000,
    lr=1e-3,
    gamma=0.995,
    tau=TAU,
    max_steps=200,
    eval_greedy=True,
    use_cache=True,
    force_retrain=False,
    cache_dir="./dqn_center_cache",
):
    os.makedirs(cache_dir, exist_ok=True)
    key = (
        f"w{center_w:.4f}_seed{int(seed)}_ep{int(center_train_episodes)}"
        f"_lr{lr}_g{gamma}_tau{tau}_ms{int(max_steps)}"
    )
    cache_path = os.path.join(cache_dir, f"center_{key}.npz")

    if use_cache and (not force_retrain) and os.path.exists(cache_path):
        data = np.load(cache_path, allow_pickle=True)
        params = data["params"].astype(np.float32)
        print(f"Loaded center pi_ref from cache: {cache_path}")
        return {
            "w": float(center_w),
            "params": params,
            "from_cache": True,
            "cache_path": cache_path,
        }

    sol = train_dqn_scalarization(
        w=float(center_w),
        seed=int(seed),
        train_episodes=center_train_episodes,
        lr=lr,
        gamma=gamma,
        tau=tau,
        max_steps=max_steps,
        eval_greedy=eval_greedy,
    )

    if use_cache:
        np.savez_compressed(cache_path, params=np.asarray(sol["params"], dtype=np.float32))
        print(f"Saved center pi_ref to cache: {cache_path}")
        sol["cache_path"] = cache_path
    sol["from_cache"] = False
    return sol


def run_equispace_weights_dqn_center_init(
    N_points,
    seed,
    center_w=0.6,
    center_train_episodes=2000,
    finetune_episodes=500,
    lr=1e-3,
    gamma=0.995,
    tau=TAU,
    max_steps=200,
    eval_greedy=True,
    use_center_cache=True,
    force_center_retrain=False,
    center_cache_dir="./dqn_center_cache",
):
    weights = np.linspace(0.0, 1.0, int(N_points))

    # 1) Long pretraining at center weight (cached).
    center_sol = get_or_train_center_dqn_solution(
        seed=int(seed + 17),
        center_w=center_w,
        center_train_episodes=center_train_episodes,
        lr=lr,
        gamma=gamma,
        tau=tau,
        max_steps=max_steps,
        eval_greedy=eval_greedy,
        use_cache=use_center_cache,
        force_retrain=force_center_retrain,
        cache_dir=center_cache_dir,
    )
    init_params = np.asarray(center_sol["params"], dtype=np.float32)

    # 2) Fine-tune all weights from the same center initialization.
    results = []
    pf = []
    for i, w in enumerate(weights):
        sol = train_dqn_scalarization(
            w=float(w),
            seed=int(seed + 40_009 * i),
            train_episodes=finetune_episodes,
            lr=lr,
            gamma=gamma,
            tau=tau,
            max_steps=max_steps,
            init_params=init_params,
            ref_params=init_params,
            eval_greedy=eval_greedy,
        )
        results.append(sol)
        pf.append([sol["R1"], sol["R2"]])

    return {
        "weights": np.asarray(weights, dtype=np.float64),
        "center_solution": center_sol,
        "results": results,
        "pf": np.asarray(pf, dtype=np.float64),
    }


def solve_cdf_refinement_dqn(
    N_points,
    seed,
    max_outer_iters=10,
    alpha=0.3,
    train_episodes=100,
    lr=1e-3,
    gamma=0.995,
    tau=TAU,
    max_steps=200,
    params0=None,
    params1=None,
    center_w=0.6,
    center_train_episodes=2000,
    eval_greedy=True,
    use_center_cache=True,
    force_center_retrain=False,
    center_cache_dir="./dqn_center_cache",
):
    quantiles = np.linspace(0.0, 1.0, int(N_points))
    fine_w = np.linspace(0.0, 1.0, 1001)
    F_vals = fine_w.copy()

    center_sol = get_or_train_center_dqn_solution(
        seed=int(seed + 7),
        center_w=center_w,
        center_train_episodes=center_train_episodes,
        lr=lr,
        gamma=gamma,
        tau=tau,
        max_steps=max_steps,
        eval_greedy=eval_greedy,
        use_cache=use_center_cache,
        force_retrain=force_center_retrain,
        cache_dir=center_cache_dir,
    )
    center_params = np.asarray(center_sol["params"], dtype=np.float32)
    current_params = [np.array(center_params, copy=True) for _ in range(int(N_points))]
    if params0 is not None and params1 is not None:
        for i, q in enumerate(quantiles):
            current_params[i] = (1.0 - q) * np.asarray(params0) + q * np.asarray(params1)

    pf_history = []
    policy_history = []
    weight_history = []

    for t in range(int(max_outer_iters)):
        current_w = np.interp(quantiles, F_vals, fine_w)
        weight_history.append(current_w.copy())

        results_t = []
        f_coords = []
        for n, w in enumerate(current_w):
            sol = train_dqn_scalarization(
                w=float(w),
                seed=int(seed + 100_003 * t + 1_009 * n),
                train_episodes=train_episodes,
                lr=lr,
                gamma=gamma,
                tau=tau,
                max_steps=max_steps,
                init_params=current_params[n],
                ref_params=center_params,
                eval_greedy=eval_greedy,
            )
            current_params[n] = np.asarray(sol["params"], dtype=np.float32)
            results_t.append(sol)
            f_coords.append([sol["R1"], sol["R2"]])

        f_coords = np.asarray(f_coords, dtype=np.float64)
        pf_history.append(f_coords)
        policy_history.append(results_t)

        diffs = np.diff(f_coords, axis=0)
        seg_lens = np.sqrt(np.sum(diffs ** 2, axis=1))
        s_vals = np.concatenate([[0.0], np.cumsum(seg_lens)])
        total_len = s_vals[-1]
        if total_len <= 1e-14:
            break

        tilde_F_values = s_vals / total_len
        interp_tilde_F = PchipInterpolator(current_w, tilde_F_values)
        tilde_vals = np.clip(interp_tilde_F(fine_w), 0.0, 1.0)

        F_vals = (1.0 - alpha) * F_vals + alpha * tilde_vals
        F_vals = np.maximum.accumulate(F_vals)
        F_vals[0], F_vals[-1] = 0.0, 1.0

    final_w = np.interp(quantiles, F_vals, fine_w)
    return {
        "final_w": final_w,
        "pf_history": pf_history,
        "policy_history": policy_history,
        "weight_history": weight_history,
        "F_grid_w": fine_w,
        "F_grid_vals": F_vals,
    }


def run_soup_weights_dqn_baseline(
    N_points,
    seed,
    endpoint_train_episodes=220,
    gamma=0.995,
    tau=TAU,
    max_steps=200,
    center_w=0.6,
    center_train_episodes=2000,
    eval_greedy=True,
    use_center_cache=True,
    force_center_retrain=False,
    center_cache_dir="./dqn_center_cache",
):
    # Pretrain center model once, then use it to initialize endpoint training.
    center_sol = get_or_train_center_dqn_solution(
        seed=int(seed + 11),
        center_w=center_w,
        center_train_episodes=center_train_episodes,
        gamma=gamma,
        tau=tau,
        max_steps=max_steps,
        eval_greedy=eval_greedy,
        use_cache=use_center_cache,
        force_retrain=force_center_retrain,
        cache_dir=center_cache_dir,
    )
    center_params = np.asarray(center_sol["params"], dtype=np.float32)

    # Train endpoints at w=0 and w=1 from center init, then linearly interpolate parameters.
    sol0 = train_dqn_scalarization(0.0, seed=seed + 17, train_episodes=endpoint_train_episodes, gamma=gamma, tau=tau, max_steps=max_steps, init_params=center_params, ref_params=center_params, eval_greedy=eval_greedy)
    sol1 = train_dqn_scalarization(1.0, seed=seed + 29, train_episodes=endpoint_train_episodes, gamma=gamma, tau=tau, max_steps=max_steps, init_params=center_params, ref_params=center_params, eval_greedy=eval_greedy)

    p0 = np.asarray(sol0["params"], dtype=np.float32)
    p1 = np.asarray(sol1["params"], dtype=np.float32)

    ws = np.linspace(0.0, 1.0, int(N_points))
    out_results = []
    out_pf = []

    model = QNet().to(DEVICE)
    for w in ws:
        p = (1.0 - float(w)) * p0 + float(w) * p1
        set_params_from_vector(model, p)
        ev = evaluate_q_policy_multiobjective(model, n_eval_episodes=16, max_steps=max_steps, gamma=gamma, tau=tau, ref_params=center_params, seed=seed + 7_001 + int(1000 * w), greedy=eval_greedy)
        ev["w"] = float(w)
        ev["params"] = np.array(p, copy=True)
        out_results.append(ev)
        out_pf.append([ev["R1"], ev["R2"]])

    return {
        "weights": ws,
        "results": out_results,
        "pf": np.asarray(out_pf, dtype=np.float64),
    }



# Default training-budget globals for OLS/UMOD / Section E (§D overwrites when that cell is run).
if "TRAIN_EP_EQUI" not in globals():
    TRAIN_EP_EQUI = 100
if "TRAIN_EP_SOUP_ENDPOINT" not in globals():
    TRAIN_EP_SOUP_ENDPOINT = 220


def run_ols_weights_dqn_baseline(
    N_points,
    seed,
    endpoint_train_episodes=None,
    train_episodes=None,
    gamma=0.995,
    tau=TAU,
    max_steps=200,
    center_w=0.6,
    center_train_episodes=2000,
    eval_greedy=True,
    use_center_cache=True,
    force_center_retrain=False,
    center_cache_dir="./dqn_center_cache",
    epsilon_ols=1e-6,
    init_mode="interp",
    duplicate_tol=1e-6,
    max_total_queries=None,
    verbose=False,
):
    """OLS (morl-baselines LinearSupport) + DQN inner solver.

    Matches Souping: same center DQN init/cache, same endpoint training at w=0,1
    for linear interpolation init, and per-OLS inner budget ``TRAIN_EP_EQUI`` (equi finetune).
    """
    if endpoint_train_episodes is None:
        endpoint_train_episodes = int(TRAIN_EP_SOUP_ENDPOINT)
    if train_episodes is None:
        train_episodes = int(TRAIN_EP_EQUI)
    try:
        from morl_baselines.multi_policy.linear_support.linear_support import LinearSupport
    except Exception as e:
        print(f"[OLS skipped] morl_baselines (or deps, e.g. pycddlib/cvxpy) unavailable: {e}")
        return None

    center_sol = get_or_train_center_dqn_solution(
        seed=int(seed + 11),
        center_w=center_w,
        center_train_episodes=center_train_episodes,
        gamma=gamma,
        tau=tau,
        max_steps=max_steps,
        eval_greedy=eval_greedy,
        use_cache=use_center_cache,
        force_retrain=force_center_retrain,
        cache_dir=center_cache_dir,
    )
    center_params = np.asarray(center_sol["params"], dtype=np.float32)

    sol0 = train_dqn_scalarization(
        0.0,
        seed=seed + 17,
        train_episodes=int(endpoint_train_episodes),
        gamma=gamma,
        tau=tau,
        max_steps=max_steps,
        init_params=center_params,
        ref_params=center_params,
        eval_greedy=eval_greedy,
    )
    sol1 = train_dqn_scalarization(
        1.0,
        seed=seed + 29,
        train_episodes=int(endpoint_train_episodes),
        gamma=gamma,
        tau=tau,
        max_steps=max_steps,
        init_params=center_params,
        ref_params=center_params,
        eval_greedy=eval_greedy,
    )
    p0 = np.asarray(sol0["params"], dtype=np.float32)
    p1 = np.asarray(sol1["params"], dtype=np.float32)

    ols = LinearSupport(num_objectives=2, epsilon=float(epsilon_ols), verbose=verbose)
    accepted_results = []
    accepted_weight_vecs = []
    prev_params = None
    num_queries = 0

    if max_total_queries is None:
        max_total_queries = max(10 * int(N_points), 50)

    def is_duplicate_solution(sol, accepted, tol=duplicate_tol):
        p = np.array(
            [sol["R1"], sol["R2"], float(sol.get("reg_term", 0.0))],
            dtype=np.float64,
        )
        for old in accepted:
            q = np.array(
                [old["R1"], old["R2"], float(old.get("reg_term", 0.0))],
                dtype=np.float64,
            )
            if np.linalg.norm(p - q, ord=2) < tol:
                return True
        return False

    while len(accepted_results) < int(N_points) and num_queries < int(max_total_queries):
        w_vec = ols.next_weight(algo="ols")
        if w_vec is None:
            break

        w_vec = np.asarray(w_vec, dtype=np.float64)
        if w_vec.sum() <= 0:
            break
        w_vec = w_vec / w_vec.sum()
        w = float(w_vec[1])

        if init_mode == "interp":
            init_params = (1.0 - w) * p0 + w * p1
        elif init_mode == "prev" and prev_params is not None:
            init_params = prev_params
        else:
            init_params = center_params.copy()

        sol = train_dqn_scalarization(
            w=w,
            seed=int(seed + 50_003 + 97 * num_queries),
            train_episodes=int(train_episodes),
            gamma=gamma,
            tau=tau,
            max_steps=max_steps,
            init_params=init_params,
            ref_params=center_params,
            eval_greedy=eval_greedy,
        )
        num_queries += 1

        reg = float(sol.get("reg_term", 0.0))
        value = np.array(
            [float(sol["R1"]) - reg, float(sol["R2"]) - reg],
            dtype=np.float64,
        )
        ols.add_solution(value=value, w=w_vec)

        if not is_duplicate_solution(sol, accepted_results, tol=duplicate_tol):
            accepted_results.append(sol)
            accepted_weight_vecs.append(w_vec.copy())
            prev_params = np.asarray(sol["params"], dtype=np.float32).copy()

        if ols.ended():
            break

    if len(accepted_results) == 0:
        return {
            "weights": np.zeros((0, 2)),
            "scalar_weights": np.zeros((0,)),
            "results": [],
            "pf": np.zeros((0, 2)),
            "num_queries": num_queries,
        }

    scalar_ws = np.array([wv[1] for wv in accepted_weight_vecs], dtype=np.float64)
    order = np.argsort(scalar_ws)
    accepted_results = [accepted_results[i] for i in order]
    accepted_weight_vecs = np.array(accepted_weight_vecs, dtype=np.float64)[order]
    scalar_ws = scalar_ws[order]

    pf = np.array([[r["R1"], r["R2"]] for r in accepted_results], dtype=np.float64)
    return {
        "weights": accepted_weight_vecs,
        "scalar_weights": scalar_ws,
        "results": accepted_results,
        "pf": pf,
        "num_queries": num_queries,
    }


def project_theta_biobj(theta_vals, eps=1e-3):
    """From DST: keep endpoints fixed, enforce separation on interior angles."""
    theta_vals = np.asarray(theta_vals, dtype=np.float64).copy()
    K = len(theta_vals)
    if K == 0:
        return theta_vals
    if K == 1:
        theta_vals[0] = 0.0
        return theta_vals
    lower = 0.0
    upper = 0.5 * np.pi
    theta_vals[0] = lower
    theta_vals[-1] = upper
    if K <= 2:
        return theta_vals
    interior = np.sort(theta_vals[1:-1])
    lo = lower + eps
    hi = upper - eps
    min_sep = eps
    interior = np.clip(interior, lo, hi)
    for i in range(1, len(interior)):
        interior[i] = max(interior[i], interior[i - 1] + min_sep)
    if len(interior) > 0 and interior[-1] > hi:
        interior[-1] = hi
        for i in range(len(interior) - 2, -1, -1):
            interior[i] = min(interior[i], interior[i + 1] - min_sep)
    interior = np.clip(interior, lo, hi)
    theta_vals[1:-1] = interior
    return theta_vals


class UMODPFModel(nn.Module):
    """PF surrogate h_phi(theta) -> R^2 (same architecture as DST notebook)."""

    def __init__(self, hidden_dim=128, out_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, theta):
        if theta.ndim == 1:
            theta = theta[:, None]
        return self.net(theta)


def theta_to_lambda_biobj(theta_vals, eps=1e-6):
    theta_vals = np.asarray(theta_vals, dtype=np.float64)
    lam1 = np.cos(theta_vals) ** 2
    lam2 = np.sin(theta_vals) ** 2
    lam = np.stack([lam1, lam2], axis=-1)
    lam = np.clip(lam, eps, None)
    lam /= np.sum(lam, axis=-1, keepdims=True)
    return lam


def lambda_to_theta_biobj(lambda_vals):
    lambda_vals = np.asarray(lambda_vals, dtype=np.float64)
    lam1 = np.clip(lambda_vals[..., 0], 0.0, 1.0)
    return np.arccos(np.sqrt(lam1))


def das_dennis_biobj(K):
    """Bi-objective Das-Dennis grid on the simplex (same as DST)."""
    t = np.linspace(0.0, 1.0, int(K), dtype=np.float64)
    return np.stack([1.0 - t, t], axis=1)


def build_initial_params_bank_from_endpoints(lambda_vals, p0=None, p1=None):
    K = len(lambda_vals)
    if p0 is None and p1 is None:
        return [None for _ in range(K)]
    if p0 is None or p1 is None:
        raise ValueError("Provide both p0 and p1, or neither.")
    p0 = np.asarray(p0, dtype=np.float32)
    p1 = np.asarray(p1, dtype=np.float32)
    bank = []
    for lam in lambda_vals:
        w = float(lam[1])
        bank.append((1.0 - w) * p0 + w * p1)
    return bank


def fit_pf_model_umod(
    theta_vals,
    pf_vals,
    hidden_dim=128,
    epochs=200,
    lr=1e-3,
    device=None,
):
    if device is None:
        device = DEVICE
    model = UMODPFModel(hidden_dim=hidden_dim, out_dim=int(pf_vals.shape[1])).to(device)
    x = torch.tensor(theta_vals, dtype=torch.float32, device=device)[:, None]
    y = torch.tensor(pf_vals, dtype=torch.float32, device=device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(int(epochs)):
        pred = model(x)
        loss = ((pred - y) ** 2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    return model


def alg_update_biobj(
    theta_vals,
    model,
    Nopt=200,
    eta=1e-2,
    device=None,
    fix_endpoints=True,
    theta_eps=1e-3,
):
    """Algorithm 2 from DST UMOD (angle update); same as Policy_update_DST_with_baseline."""
    if device is None:
        device = DEVICE
    theta_vals = project_theta_biobj(theta_vals, eps=theta_eps)
    theta = torch.tensor(theta_vals, dtype=torch.float32, device=device)
    lower = 0.0
    upper = 0.5 * np.pi
    for _ in range(int(Nopt)):
        with torch.no_grad():
            pred = model(theta[:, None])
            D = torch.cdist(pred, pred, p=2)
            K = D.shape[0]
            D.fill_diagonal_(float("inf"))
            flat_idx = torch.argmin(D)
            i_star = int(flat_idx // K)
            j_star = int(flat_idx % K)
        ti = theta[i_star].detach().clone().requires_grad_(True)
        tj = theta[j_star].detach().clone().requires_grad_(True)
        yi = model(ti.view(1, 1)).view(-1)
        yj = model(tj.view(1, 1)).view(-1)
        rho = torch.norm(yi - yj, p=2).clamp_min(1e-12)
        direction = ((yi - yj) / rho).detach()
        grad_i = torch.autograd.grad(yi, ti, grad_outputs=direction, retain_graph=False, create_graph=False)[0]
        grad_j = torch.autograd.grad(yj, tj, grad_outputs=direction, retain_graph=False, create_graph=False)[0]
        with torch.no_grad():
            i_fixed = fix_endpoints and (i_star == 0 or i_star == K - 1)
            j_fixed = fix_endpoints and (j_star == 0 or j_star == K - 1)
            if not i_fixed:
                theta[i_star] = torch.clamp(theta[i_star] + eta * grad_i, lower, upper)
            if not j_fixed:
                theta[j_star] = torch.clamp(theta[j_star] - eta * grad_j, lower, upper)
            if fix_endpoints and len(theta) >= 2:
                theta_np = theta.detach().cpu().numpy()
                theta_np = project_theta_biobj(theta_np, eps=theta_eps)
                theta.copy_(torch.tensor(theta_np, dtype=theta.dtype, device=device))
    return theta.detach().cpu().numpy()


def run_umod_weights_dqn_baseline(
    N_points,
    seed,
    endpoint_train_episodes=None,
    train_episodes=None,
    gamma=0.995,
    tau=TAU,
    max_steps=200,
    center_w=0.6,
    center_train_episodes=2000,
    max_outer_iters=10,
    pf_model_hidden=128,
    pf_model_epochs=200,
    pf_model_lr=1e-3,
    Nopt=200,
    theta_step=1e-2,
    theta_eps=5e-3,
    eval_greedy=True,
    use_center_cache=True,
    force_center_retrain=False,
    center_cache_dir="./dqn_center_cache",
    verbose=False,
):
    """Faithful UMOD outer loop (DST); inner oracle = DQN linear scalarization (same as CDF / equi / soup)."""
    if endpoint_train_episodes is None:
        endpoint_train_episodes = int(TRAIN_EP_SOUP_ENDPOINT)
    if train_episodes is None:
        train_episodes = int(TRAIN_EP_EQUI)

    center_sol = get_or_train_center_dqn_solution(
        seed=int(seed + 11),
        center_w=center_w,
        center_train_episodes=center_train_episodes,
        gamma=gamma,
        tau=tau,
        max_steps=max_steps,
        eval_greedy=eval_greedy,
        use_cache=use_center_cache,
        force_retrain=force_center_retrain,
        cache_dir=center_cache_dir,
    )
    center_params = np.asarray(center_sol["params"], dtype=np.float32)

    sol0 = train_dqn_scalarization(
        0.0,
        seed=seed + 17,
        train_episodes=int(endpoint_train_episodes),
        gamma=gamma,
        tau=tau,
        max_steps=max_steps,
        init_params=center_params,
        ref_params=center_params,
        eval_greedy=eval_greedy,
    )
    sol1 = train_dqn_scalarization(
        1.0,
        seed=seed + 29,
        train_episodes=int(endpoint_train_episodes),
        gamma=gamma,
        tau=tau,
        max_steps=max_steps,
        init_params=center_params,
        ref_params=center_params,
        eval_greedy=eval_greedy,
    )
    p0 = np.asarray(sol0["params"], dtype=np.float32)
    p1 = np.asarray(sol1["params"], dtype=np.float32)

    lambda_vals = das_dennis_biobj(N_points)
    theta_vals = lambda_to_theta_biobj(lambda_vals)
    theta_vals = project_theta_biobj(theta_vals, eps=theta_eps)
    params_bank = build_initial_params_bank_from_endpoints(lambda_vals, p0, p1)

    history = []
    pf_model = None

    for outer_it in range(int(max_outer_iters)):
        theta_vals = project_theta_biobj(theta_vals, eps=theta_eps)
        lambda_vals = theta_to_lambda_biobj(theta_vals)

        results_round = []
        next_bank = []
        pf_reg_rows = []

        for k in range(int(N_points)):
            w = float(lambda_vals[k, 1])
            sol = train_dqn_scalarization(
                w=w,
                seed=int(seed + 80_003 * outer_it + 503 * k),
                train_episodes=int(train_episodes),
                gamma=gamma,
                tau=tau,
                max_steps=max_steps,
                init_params=params_bank[k],
                ref_params=center_params,
                eval_greedy=eval_greedy,
            )
            sol["theta"] = float(theta_vals[k])
            sol["lambda"] = lambda_vals[k].copy()
            sol["w"] = w
            results_round.append(sol)
            next_bank.append(np.asarray(sol["params"], dtype=np.float32).copy())
            reg = float(sol.get("reg_term", 0.0))
            pf_reg_rows.append([float(sol["R1"]) - reg, float(sol["R2"]) - reg])

        pf_reg = np.asarray(pf_reg_rows, dtype=np.float64)

        pf_model = fit_pf_model_umod(
            theta_vals,
            pf_reg,
            hidden_dim=int(pf_model_hidden),
            epochs=int(pf_model_epochs),
            lr=float(pf_model_lr),
            device=DEVICE,
        )

        history.append(
            {
                "outer_it": outer_it,
                "theta": theta_vals.copy(),
                "lambda": lambda_vals.copy(),
                "pf_reg_reward": pf_reg.copy(),
                "results": results_round,
            }
        )

        if verbose:
            dmat = np.linalg.norm(pf_reg[:, None, :] - pf_reg[None, :, :], axis=-1)
            np.fill_diagonal(dmat, np.inf)
            min_gap = float(np.min(dmat))
            print(
                f"[UMOD-DQN] outer={outer_it:02d} K={int(N_points)} "
                f"min_pairwise_gap={min_gap:.6e}"
            )

        theta_vals = alg_update_biobj(
            theta_vals=theta_vals,
            model=pf_model,
            Nopt=int(Nopt),
            eta=float(theta_step),
            device=DEVICE,
            fix_endpoints=True,
            theta_eps=float(theta_eps),
        )
        params_bank = next_bank

    theta_vals = project_theta_biobj(theta_vals, eps=theta_eps)
    final_lambda = theta_to_lambda_biobj(theta_vals)
    final_results = []
    for k in range(int(N_points)):
        w = float(final_lambda[k, 1])
        sol = train_dqn_scalarization(
            w=w,
            seed=int(seed + 99_003 + 733 * k),
            train_episodes=int(train_episodes),
            gamma=gamma,
            tau=tau,
            max_steps=max_steps,
            init_params=params_bank[k],
            ref_params=center_params,
            eval_greedy=eval_greedy,
        )
        sol["theta"] = float(theta_vals[k])
        sol["lambda"] = final_lambda[k].copy()
        sol["w"] = w
        final_results.append(sol)

    order = np.argsort(theta_vals)
    theta_sorted = theta_vals[order]
    results_sorted = [final_results[i] for i in order]
    weights_sorted = final_lambda[order]
    scalar_w = weights_sorted[:, 1].astype(np.float64)
    pf = np.array([[r["R1"], r["R2"]] for r in results_sorted], dtype=np.float64)

    return {
        "theta": theta_sorted,
        "weights": weights_sorted,
        "scalar_weights": scalar_w,
        "results": results_sorted,
        "pf": pf,
        "history": history,
        "pf_model": pf_model,
    }



def run_warmstart_adaptive_dqn_baseline(
    N_points,
    seed,
    rounds=3,
    train_episodes=120,
    gamma=0.995,
    tau=TAU,
    max_steps=200,
    center_w=0.6,
    center_train_episodes=2000,
    eval_greedy=True,
    use_center_cache=True,
    force_center_retrain=False,
    center_cache_dir="./dqn_center_cache",
):
    ws = np.linspace(0.0, 1.0, int(N_points))
    last_results = None

    center_sol = get_or_train_center_dqn_solution(
        seed=int(seed + 13),
        center_w=center_w,
        center_train_episodes=center_train_episodes,
        gamma=gamma,
        tau=tau,
        max_steps=max_steps,
        eval_greedy=eval_greedy,
        use_cache=use_center_cache,
        force_retrain=force_center_retrain,
        cache_dir=center_cache_dir,
    )
    center_params = np.asarray(center_sol["params"], dtype=np.float32)
    params_by_idx = [np.array(center_params, copy=True) for _ in range(int(N_points))]

    for r in range(int(rounds)):
        cur_results = []
        pf = []
        for i, w in enumerate(ws):
            sol = train_dqn_scalarization(
                w=float(w),
                seed=int(seed + 20_003 * r + 211 * i),
                train_episodes=train_episodes,
                gamma=gamma,
                tau=tau,
                max_steps=max_steps,
                init_params=params_by_idx[i],
                ref_params=center_params,
                eval_greedy=eval_greedy,
            )
            params_by_idx[i] = np.asarray(sol["params"], dtype=np.float32)
            cur_results.append(sol)
            pf.append([sol["R1"], sol["R2"]])

        pf = np.asarray(pf, dtype=np.float64)
        last_results = cur_results

        if len(pf) < 3:
            continue
        seg = np.sqrt(np.sum(np.diff(pf, axis=0) ** 2, axis=1))
        dense = np.ones_like(ws)
        dense[1:-1] = 0.5 * (seg[:-1] + seg[1:])
        c = np.cumsum(np.maximum(dense, 1e-9))
        c = (c - c[0]) / (c[-1] - c[0] + 1e-12)
        ws = np.interp(np.linspace(0.0, 1.0, int(N_points)), c, ws)

    pf_out = np.array([[s["R1"], s["R2"]] for s in last_results], dtype=np.float64)
    return {
        "weights": np.asarray(ws, dtype=np.float64),
        "results": last_results,
        "pf": pf_out,
    }


## C. Metrics and reporting

In [14]:

def front_segment_lengths(pf):
    pf = np.asarray(pf, dtype=float)
    if len(pf) < 2:
        return np.array([])
    return np.sqrt(np.sum(np.diff(pf, axis=0) ** 2, axis=1))


def arc_length_cv(pf):
    segs = front_segment_lengths(pf)
    segs = segs[segs > 1e-15]
    if len(segs) < 2:
        return np.nan
    mu = np.mean(segs)
    if mu <= 1e-15:
        return np.nan
    return float(np.std(segs, ddof=0) / mu)


def max_min_gap_ratio(pf):
    segs = front_segment_lengths(pf)
    segs = segs[segs > 1e-15]
    if len(segs) == 0:
        return np.nan
    return float(np.max(segs) / np.min(segs))


def calculate_relative_rms_err(pf):
    pf = np.asarray(pf, dtype=float)
    if len(pf) < 2:
        return np.nan
    seg_lens = np.sqrt(np.sum(np.diff(pf, axis=0) ** 2, axis=1))
    seg_lens = seg_lens[seg_lens > 1e-15]
    if len(seg_lens) < 2:
        return np.nan
    L = np.sum(seg_lens)
    if L < 1e-12:
        return 0.0
    ideal_len = L / len(seg_lens)
    relative_deviations = (seg_lens / ideal_len) - 1.0
    return float(np.sqrt(np.mean(relative_deviations ** 2)))


def igd(pf, ref_pf):
    pf = np.asarray(pf, dtype=float)
    ref_pf = np.asarray(ref_pf, dtype=float)
    if len(pf) == 0 or len(ref_pf) == 0:
        return np.nan
    dists = np.linalg.norm(ref_pf[:, None, :] - pf[None, :, :], axis=2)
    return float(np.mean(np.min(dists, axis=1)))


def compute_hv(objs, ref_point):
    objs = np.asarray(objs, dtype=float)
    if len(objs) == 0:
        return 0.0
    objs = objs[np.argsort(objs[:, 0])]
    x, hv = ref_point[0], 0.0
    for i in range(len(objs)):
        hv += (max(ref_point[0], objs[i, 0]) - x) * (max(ref_point[1], objs[i, 1]) - ref_point[1])
        x = max(ref_point[0], objs[i, 0])
    return float(hv)


def expected_utility_per_solution(results):
    vals = []
    for s in results:
        w = float(s["w"])
        vals.append((1.0 - w) * float(s["R1"]) + w * float(s["R2"]))
    return np.asarray(vals, dtype=float)


def _solution_episode_return(result, env_name="mo-mountaincar-v0", horizon=200, gamma=0.995, base_seed=0):
    """One stochastic rollout return vector for a single solution dict containing Q-network params."""
    if "params" not in result:
        return np.array([np.nan, np.nan], dtype=np.float64)

    model = QNet(hidden=128).to(DEVICE)
    set_params_from_vector(model, np.asarray(result["params"], dtype=np.float32))
    model.eval()

    env = make_env(env_name, max_episode_steps=horizon)
    state = reset_env(env, seed=int(base_seed))
    g = np.zeros(2, dtype=np.float64)
    disc = 1.0

    for _ in range(int(horizon)):
        s = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            q = model(s).squeeze(0)
            probs = torch.softmax(q, dim=0)
            action = int(torch.distributions.Categorical(probs=probs).sample().item())

        state, env_reward, done, _, _, _ = step_env(env, action)
        rv = vector_reward_from_step(env_reward)
        g += disc * rv
        disc *= gamma
        if done:
            break

    env.close()
    return g


def episodic_dominance_pairwise(results_a, results_b, n_episodes=128, horizon=200, gamma=0.995, base_seed=0):
    """ED_{A,B} = E_{lambda~U(0,1)}[ 1{ u_A > u_B } ], u=omega^T G with omega=(lambda,1-lambda)."""
    if len(results_a) == 0 or len(results_b) == 0:
        return np.nan

    w_a = np.array([float(s.get("w", np.nan)) for s in results_a], dtype=np.float64)
    w_b = np.array([float(s.get("w", np.nan)) for s in results_b], dtype=np.float64)
    if np.any(~np.isfinite(w_a)) or np.any(~np.isfinite(w_b)):
        return np.nan

    wins = 0
    for ep in range(int(n_episodes)):
        rng = np.random.default_rng(int(base_seed) + ep * 7919 + 17)
        lam = float(rng.uniform(0.0, 1.0))
        omega = np.array([lam, 1.0 - lam], dtype=np.float64)

        ia = int(np.argmin(np.abs(w_a - lam)))
        ib = int(np.argmin(np.abs(w_b - lam)))

        g_a = _solution_episode_return(results_a[ia], horizon=horizon, gamma=gamma, base_seed=int(base_seed) + 100_003 * ep)
        g_b = _solution_episode_return(results_b[ib], horizon=horizon, gamma=gamma, base_seed=int(base_seed) + 100_003 * ep + 50_000)

        if not (np.all(np.isfinite(g_a)) and np.all(np.isfinite(g_b))):
            continue

        if float(omega @ g_a) > float(omega @ g_b):
            wins += 1

    return float(wins) / float(n_episodes)


def mean_pairwise_episodic_dominance(all_results, method_name, n_episodes=128, horizon=200, gamma=0.995, base_seed=42):
    """Average directed ED(method, other) over all other methods."""
    if method_name not in all_results:
        return np.nan
    others = [n for n in all_results.keys() if n != method_name]
    if not others:
        return np.nan

    vals = []
    for opp in others:
        seed = int(base_seed + (abs(hash((method_name, opp))) % 1_000_000))
        v = episodic_dominance_pairwise(
            all_results[method_name], all_results[opp],
            n_episodes=n_episodes, horizon=horizon, gamma=gamma, base_seed=seed,
        )
        if np.isfinite(v):
            vals.append(v)

    return float(np.mean(vals)) if vals else np.nan


# Episodic Dominance config
ED_N_EPISODES = 128
ED_HORIZON = 200
ED_SEED = 42


def pareto_metrics_vs_ref(pf, ref_pf, hv_ref):
    return {
        "Err_arc": calculate_relative_rms_err(pf),
        "CV": arc_length_cv(pf),
        "GapRatio_m1": np.abs(max_min_gap_ratio(pf) - 1.0),
        "IGD": igd(pf, ref_pf),
        "HV": compute_hv(pf, hv_ref),
    }


def mean_std_finite(vals):
    a = np.asarray(vals, dtype=float)
    a = a[np.isfinite(a)]
    if len(a) == 0:
        return np.nan, np.nan
    if len(a) == 1:
        return float(a[0]), 0.0
    return float(np.mean(a)), float(np.std(a, ddof=1))



## D. Single-seed run and PF visualization

In [ ]:

# Runtime knobs (increase train_episodes for stronger policies)
N_sparse = 6
N_ref = 201
BASE_SEED = 0

print("TAU (KL toward center pi_ref):", TAU)
EVAL_GREEDY = True  # set True to diagnose argmax-collapse behavior
print("Evaluation mode:", "greedy (argmax)" if EVAL_GREEDY else "stochastic (sampling)")

TRAIN_EP_DQN = 1000
TRAIN_EP_DQN_CENTER = 2000
TRAIN_EP_DQN_FINETUNE = 1000
TRAIN_EP_CDF = 100
TRAIN_EP_UMOD = 90
TRAIN_EP_SOUP_ENDPOINT = 220
TAU_DQN = 0.05  # KL regularization strength for DQN inner solver
# Match equi finetune in Section E to the single-run DQN demo (fair comparison with CDF / Souping / OLS / UMOD).
TRAIN_EP_EQUI = TRAIN_EP_DQN_FINETUNE
GAMMA_DQN = 1.0  # same as out_dqn_equi / out_dqn_cdf below; Section E uses this for all DQN baselines

set_all_seeds(BASE_SEED)

# Endpoint models for initialization
# sol_w0 = train_policy_scalarization(0.0, seed=BASE_SEED + 101, train_episodes=TRAIN_EP_SOUP_ENDPOINT, max_steps=200, eval_greedy=EVAL_GREEDY)
# sol_w1 = train_policy_scalarization(1.0, seed=BASE_SEED + 202, train_episodes=TRAIN_EP_SOUP_ENDPOINT, max_steps=200, eval_greedy=EVAL_GREEDY)
# params0, params1 = sol_w0["params"], sol_w1["params"]

# Train all DQN methods
out_dqn_equi = run_equispace_weights_dqn_center_init(
    N_sparse,
    seed=BASE_SEED + 450,
    center_w=0.6,
    center_train_episodes=TRAIN_EP_DQN_CENTER,
    finetune_episodes=TRAIN_EP_DQN_FINETUNE,
    gamma=GAMMA_DQN,
    tau=TAU_DQN,
    max_steps=200,
    eval_greedy=True,
)

# out_dqn_equi = run_equispace_weights_dqn(
#     N_sparse,
#     seed=BASE_SEED + 450,
#     train_episodes=TRAIN_EP_DQN,
#     gamma=0.995,
#     tau=TAU_DQN,
#     max_steps=200,
#     eval_greedy=True,
# )
out_dqn_cdf = solve_cdf_refinement_dqn(
    N_sparse,
    seed=BASE_SEED + 500,
    max_outer_iters=10,
    alpha=0.3,
    train_episodes=TRAIN_EP_CDF,
    center_train_episodes=TRAIN_EP_DQN_CENTER,
    gamma=GAMMA_DQN,
    tau=TAU_DQN,
    max_steps=200,
    eval_greedy=True,
)
# out_dqn_soup = run_soup_weights_dqn_baseline(
#     N_sparse,
#     seed=BASE_SEED + 600,
#     endpoint_train_episodes=TRAIN_EP_SOUP_ENDPOINT,
#     max_steps=200,
#     eval_greedy=True,
# )
# out_dqn_umod = run_warmstart_adaptive_dqn_baseline(
#     N_sparse,
#     seed=BASE_SEED + 700,
#     rounds=3,
#     train_episodes=TRAIN_EP_UMOD,
#     max_steps=200,
#     eval_greedy=True,
# )

# Cache PF arrays for the visualization cell
pf_dqn_equi = out_dqn_equi["pf"]
pf_dqn_cdf = out_dqn_cdf["pf_history"][-1]
# pf_dqn_soup = out_dqn_soup["pf"]
# pf_dqn_umod = out_dqn_umod["pf"]

# Sanity check: loss vs epochs (separate curves per weight)
loss_pairs = []
# for r in out_equi.get("results", []):
#     curve = np.asarray(r.get("loss_history", []), dtype=np.float32)
#     if curve.size > 0:
#         loss_pairs.append((float(r.get("w", np.nan)), curve))

if not loss_pairs:
    print("No loss history found. Re-run helper cell definitions first.")
else:
    if len(loss_pairs) > 5:
        idxs = np.linspace(0, len(loss_pairs) - 1, 5, dtype=int)
        loss_pairs = [loss_pairs[i] for i in idxs]

    epoch_counts = [int(curve.size) for _, curve in loss_pairs]
    print(f"Epochs trained per plotted policy: {epoch_counts}")
    print(f"Min/Max epochs: {min(epoch_counts)}/{max(epoch_counts)}")

    min_len = int(min(curve.size for _, curve in loss_pairs))
    epochs = np.arange(1, min_len + 1)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for w_val, curve in loss_pairs:
        ax.plot(epochs, curve[:min_len], lw=1.8, label=f"w={w_val:.2f}")

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Sanity Check: Loss vs Epochs (5 weights)")
    ax.legend(loc="best", ncol=2)
    plt.show()

# Sanity check: scalarized objective vs epochs (separate curves per weight)
obj_pairs = []
# for r in out_equi.get("results", []):
#     curve = np.asarray(r.get("scalar_obj_history", []), dtype=np.float32)
#     if curve.size > 0:
#         obj_pairs.append((float(r.get("w", np.nan)), curve))

if not obj_pairs:
    print("No scalarized-objective history found. Re-run helper cell definitions first.")
else:
    # Show five representative weights by default; if exactly five are present, show all.
    if len(obj_pairs) > 5:
        idxs = np.linspace(0, len(obj_pairs) - 1, 5, dtype=int)
        obj_pairs = [obj_pairs[i] for i in idxs]

    min_len_obj = int(min(curve.size for _, curve in obj_pairs))
    epochs_obj = np.arange(1, min_len_obj + 1)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for w_val, curve in obj_pairs:
        ax.plot(epochs_obj, curve[:min_len_obj], lw=1.8, label=f"w={w_val:.2f}")

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Scalarized objective")
    ax.set_title("Sanity Check (PG): Scalarized Objective vs Epochs (5 weights)")
    ax.legend(loc="best", ncol=2)
    plt.show()

# DQN sanity check: loss vs updates (separate curves per weight)
dqn_loss_pairs = []
for r in out_dqn_equi.get("results", []):
    curve = np.asarray(r.get("loss_history", []), dtype=np.float32)
    if curve.size > 0:
        dqn_loss_pairs.append((float(r.get("w", np.nan)), curve))

if not dqn_loss_pairs:
    print("No DQN loss history found. Re-run DQN training cell.")
else:
    if len(dqn_loss_pairs) > 5:
        idxs = np.linspace(0, len(dqn_loss_pairs) - 1, 5, dtype=int)
        dqn_loss_pairs = [dqn_loss_pairs[i] for i in idxs]

    min_len = int(min(curve.size for _, curve in dqn_loss_pairs))
    x = np.arange(1, min_len + 1)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for w_val, curve in dqn_loss_pairs:
        ax.plot(x, curve[:min_len], lw=1.8, label=f"w={w_val:.2f}")

    ax.set_xlabel("Update step")
    ax.set_ylabel("TD loss")
    ax.set_title("Sanity Check (DQN): Loss vs Updates (5 weights)")
    ax.legend(loc="best", ncol=2)
    plt.show()

# DQN sanity check: scalarized objective vs epochs (separate curves per weight)
dqn_obj_pairs = []
for r in out_dqn_equi.get("results", []):
    curve = np.asarray(r.get("scalar_obj_history", []), dtype=np.float32)
    if curve.size > 0:
        dqn_obj_pairs.append((float(r.get("w", np.nan)), curve))

if not dqn_obj_pairs:
    print("No DQN scalarized-objective history found. Re-run DQN training cell.")
else:
    if len(dqn_obj_pairs) > 5:
        idxs = np.linspace(0, len(dqn_obj_pairs) - 1, 5, dtype=int)
        dqn_obj_pairs = [dqn_obj_pairs[i] for i in idxs]

    min_len = int(min(curve.size for _, curve in dqn_obj_pairs))
    x = np.arange(1, min_len + 1)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for w_val, curve in dqn_obj_pairs:
        ax.plot(x, curve[:min_len], lw=1.8, label=f"w={w_val:.2f}")

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Scalarized objective")
    ax.set_title("Sanity Check (DQN): Scalarized Objective vs Epochs (5 weights)")
    ax.legend(loc="best", ncol=2)
    plt.show()


In [ ]:
# PF visualization (run after the DQN training cell above)
fig, ax = plt.subplots(figsize=(7, 5))
# ax.plot(pf_ref[:, 0], pf_ref[:, 1], color="0.75", lw=3, label=f"Reference equi-w (N={N_ref})")
# ax.scatter(pf_equi[:, 0], pf_equi[:, 1], s=60, marker="s", label="Equal-spaced (PG)")
ax.scatter(pf_dqn_equi[:, 0], pf_dqn_equi[:, 1], s=60, marker="^", label="DQN Equal-spaced")
# print("PG PF:\n", pf_equi)
print("DQN Equal-spaced PF:\n", pf_dqn_equi)
print("DQN CDF PF:\n", pf_dqn_cdf)
# print("DQN Soup PF:\n", pf_dqn_soup)
# print("DQN Adaptive PF:\n", pf_dqn_umod)
ax.scatter(pf_dqn_cdf[:, 0], pf_dqn_cdf[:, 1], s=60, marker="o", label="DQN CDF refinement")
# ax.scatter(pf_dqn_soup[:, 0], pf_dqn_soup[:, 1], s=55, marker="x", label="DQN Souping")
# ax.scatter(pf_dqn_umod[:, 0], pf_dqn_umod[:, 1], s=55, marker="d", label="DQN Adaptive warm-start")
ax.set_xlabel("Objective 1: time penalty - tau*KL (discounted)")
ax.set_ylabel("Objective 2: reverse penalty - tau*KL (discounted)")
ax.set_title("MountainCar PF coverage (all DQN variants, regularized)")
ax.legend(loc="best")
plt.show()

# Scalarized rewards vs epochs (5 weights)
scalar_pairs = []
for r in out_dqn_equi.get("results", []):
    c = np.asarray(r.get("scalar_obj_history", []), dtype=np.float32)
    if c.size > 0:
        scalar_pairs.append((float(r.get("w", np.nan)), c))
if scalar_pairs:
    if len(scalar_pairs) > 5:
        idxs = np.linspace(0, len(scalar_pairs) - 1, 5, dtype=int)
        scalar_pairs = [scalar_pairs[i] for i in idxs]
    m = int(min(c.size for _, c in scalar_pairs))
    x = np.arange(1, m + 1)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for wv, c in scalar_pairs:
        ax.plot(x, c[:m], lw=1.8, label=f"w={wv:.2f}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Scalarized reward")
    ax.set_title("DQN Scalarized Rewards vs Epochs")
    ax.legend(loc="best", ncol=2)
    plt.show()

# -tau*KL vs epochs (5 weights)
kl_pairs = []
for r in out_dqn_equi.get("results", []):
    c = np.asarray(r.get("neg_kl_tau_history", []), dtype=np.float32)
    if c.size > 0:
        kl_pairs.append((float(r.get("w", np.nan)), c))
if kl_pairs:
    if len(kl_pairs) > 5:
        idxs = np.linspace(0, len(kl_pairs) - 1, 5, dtype=int)
        kl_pairs = [kl_pairs[i] for i in idxs]
    m = int(min(c.size for _, c in kl_pairs))
    x = np.arange(1, m + 1)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for wv, c in kl_pairs:
        ax.plot(x, c[:m], lw=1.8, label=f"w={wv:.2f}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("-tau * KL")
    ax.set_title("DQN -tau*KL vs Epochs")
    ax.legend(loc="best", ncol=2)
    plt.show()

# PF visualization without KL regularizer (raw rewards)
pf_dqn_equi_raw = np.array([[r["R1_raw"], r["R2_raw"]] for r in out_dqn_equi["results"]], dtype=np.float64)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(pf_dqn_equi_raw[:, 0], pf_dqn_equi_raw[:, 1], s=60, marker="^", label="DQN Equal-spaced (raw)")
print("DQN Equal-spaced PF (raw):\n", pf_dqn_equi_raw)
ax.set_xlabel("R1")
ax.set_ylabel("R2")
ax.set_title("MountainCar PF coverage (without KL regularizer)")
ax.legend(loc="best")
plt.show()

In [ ]:

# # Runtime knobs (increase train_episodes for stronger policies)
# N_sparse = 6
# N_ref = 201
# BASE_SEED = 0

# print("TAU (KL toward center pi_ref):", TAU)
# EVAL_GREEDY = False  # set True to diagnose argmax-collapse behavior
# print("Evaluation mode:", "greedy (argmax)" if EVAL_GREEDY else "stochastic (sampling)")

# TRAIN_EP_EQUI = 1000
# TRAIN_EP_DQN = 1000
# TRAIN_EP_DQN_CENTER = 2000
# TRAIN_EP_DQN_FINETUNE = 500
# TRAIN_EP_CDF = 1000
# TRAIN_EP_UMOD = 1000
# TRAIN_EP_SOUP_ENDPOINT = 1000
# TAU_DQN = 1.5  # KL regularization strength for DQN inner solver

# set_all_seeds(BASE_SEED)

# # Endpoint models for initialization (not needed for pure DQN run)
# # sol_w0 = train_policy_scalarization(0.0, seed=BASE_SEED + 101, train_episodes=TRAIN_EP_SOUP_ENDPOINT, max_steps=200, eval_greedy=EVAL_GREEDY)
# # sol_w1 = train_policy_scalarization(1.0, seed=BASE_SEED + 202, train_episodes=TRAIN_EP_SOUP_ENDPOINT, max_steps=200, eval_greedy=EVAL_GREEDY)
# # params0, params1 = sol_w0["params"], sol_w1["params"]

# # Train all DQN methods
# out_dqn_equi = run_equispace_weights_dqn_center_init(
#     N_sparse,
#     seed=BASE_SEED + 450,
#     center_w=0.5,
#     center_train_episodes=TRAIN_EP_DQN_CENTER,
#     finetune_episodes=TRAIN_EP_DQN_FINETUNE,
#     tau=TAU_DQN,
#     max_steps=200,
#     eval_greedy=True,
# )
# out_dqn_cdf = solve_cdf_refinement_dqn(
#     N_sparse,
#     seed=BASE_SEED + 500,
#     max_outer_iters=6,
#     alpha=0.5,
#     train_episodes=TRAIN_EP_CDF,
#     tau=TAU_DQN,
#     max_steps=200,
#     center_w=0.5,
#     center_train_episodes=TRAIN_EP_DQN_CENTER,
#     eval_greedy=True,
# )
# out_dqn_soup = run_soup_weights_dqn_baseline(
#     N_sparse,
#     seed=BASE_SEED + 600,
#     endpoint_train_episodes=TRAIN_EP_SOUP_ENDPOINT,
#     tau=TAU_DQN,
#     max_steps=200,
#     center_w=0.5,
#     center_train_episodes=TRAIN_EP_DQN_CENTER,
#     eval_greedy=True,
# )
# out_dqn_umod = run_warmstart_adaptive_dqn_baseline(
#     N_sparse,
#     seed=BASE_SEED + 700,
#     rounds=3,
#     train_episodes=TRAIN_EP_UMOD,
#     tau=TAU_DQN,
#     max_steps=200,
#     center_w=0.5,
#     center_train_episodes=TRAIN_EP_DQN_CENTER,
#     eval_greedy=True,
# )

# # Cache PF arrays for the visualization cell
# pf_dqn_equi = out_dqn_equi["pf"]
# pf_dqn_cdf = out_dqn_cdf["pf_history"][-1]
# pf_dqn_soup = out_dqn_soup["pf"]
# pf_dqn_umod = out_dqn_umod["pf"]

# print("Training done. Run the next cell to visualize PF coverage.")


## E. Seed-wise comparison table (mean/std across seeds only)

Run **§D** first so `TRAIN_EP_*`, `GAMMA_DQN`, and `TAU_DQN` match the single-seed demo. Subsections: **E.1** cohort table below; **Souping**; **E.2 OLS**; **E.3 UMOD**; then the filtered re-display cell.

In [ ]:
ALGO_SEEDS = [0, 3, 12]
# ALGO_SEEDS = [0]
method_metric_series = {}
seed_metric_rows = []

metric_keys = [
    "Err_arc", "CV", "GapRatio_m1", "IGD", "HV"
]

if "TRAIN_EP_CDF" not in globals():
    TRAIN_EP_CDF = 100
if "TRAIN_EP_EQUI" not in globals():
    TRAIN_EP_EQUI = 100

# Same DQN hyperparameters as §D (GAMMA_DQN, TAU_DQN, TRAIN_EP_DQN_CENTER) when that cell was run.
_DQN_GAMMA = GAMMA_DQN if "GAMMA_DQN" in globals() else 0.995
_DQN_TAU = TAU_DQN if "TAU_DQN" in globals() else TAU
_DQN_CENTER_EP = int(TRAIN_EP_DQN_CENTER) if "TRAIN_EP_DQN_CENTER" in globals() else 2000

for si, seed in enumerate(ALGO_SEEDS):
    set_all_seeds(seed)
    seed_rows_current = []

    # Keep each seed independent by offsetting internal seeds
    out_dense_s = run_equispace_weights_dqn_center_init(
        N_ref,
        seed=10_000 + seed,
        center_train_episodes=_DQN_CENTER_EP,
        finetune_episodes=TRAIN_EP_EQUI,
        gamma=_DQN_GAMMA,
        tau=_DQN_TAU,
        max_steps=200,
        eval_greedy=EVAL_GREEDY,
    )
    ref_pf = np.asarray(out_dense_s["pf"], dtype=np.float64)

    out_equi_s = run_equispace_weights_dqn_center_init(
        N_sparse,
        seed=20_000 + seed,
        center_train_episodes=_DQN_CENTER_EP,
        finetune_episodes=TRAIN_EP_EQUI,
        gamma=_DQN_GAMMA,
        tau=_DQN_TAU,
        max_steps=200,
        eval_greedy=EVAL_GREEDY,
    )
    out_cdf_s = solve_cdf_refinement_dqn(
        N_sparse,
        seed=30_000 + seed,
        max_outer_iters=10,
        alpha=0.3,
        train_episodes=TRAIN_EP_CDF,
        center_train_episodes=_DQN_CENTER_EP,
        gamma=_DQN_GAMMA,
        tau=_DQN_TAU,
        max_steps=200,
        eval_greedy=EVAL_GREEDY,
    )

    all_pf_s = {
        "Equal-spaced": np.asarray(out_equi_s["pf"], dtype=np.float64),
        "CDF": np.asarray(out_cdf_s["pf_history"][-1], dtype=np.float64),
        # "Souping": np.asarray(out_soup_s["pf"], dtype=np.float64),
        # "AdaptiveWarm": np.asarray(out_umod_s["pf"], dtype=np.float64),
    }
    all_results_s = {
        "Equal-spaced": out_equi_s["results"],
        "CDF": out_cdf_s["policy_history"][-1],
        # "Souping": out_soup_s["results"],
        # "AdaptiveWarm": out_umod_s["results"],
    }

    all_pts = np.vstack(list(all_pf_s.values()))
    hv_ref = np.array([np.min(all_pts[:, 0]) - 1.0, np.min(all_pts[:, 1]) - 1.0], dtype=np.float64)

    for name, pf in all_pf_s.items():
        m = pareto_metrics_vs_ref(pf, ref_pf, hv_ref)
        m["MeanUtility"] = float(np.mean(expected_utility_per_solution(all_results_s[name])))

        # Aggregate for mean/std
        if name not in method_metric_series:
            method_metric_series[name] = {k: [] for k in m.keys()}
        for k, v in m.items():
            method_metric_series[name][k].append(v)

        # Per-seed reporting row
        row_seed = {"Seed": int(seed), "Method": name}
        for k in metric_keys:
            row_seed[k] = float(m.get(k, np.nan))
        seed_metric_rows.append(row_seed)
        seed_rows_current.append(row_seed)

    print(f"\n[Seed {seed}] metrics")
    for r in sorted(seed_rows_current, key=lambda x: x["Method"]):
        print(f"  Method={r['Method']}")
        for k in metric_keys:
            v = r[k]
            print(f"    {k}: {v:.6g}" if np.isfinite(v) else f"    {k}: nan")

# -------- Per-seed report --------
try:
    import pandas as pd
    seed_cols = ["Seed", "Method"] + metric_keys
    per_seed_df = pd.DataFrame(seed_metric_rows)[seed_cols].sort_values(["Seed", "Method"]).reset_index(drop=True)
    print("Per-seed metrics:")
    display(per_seed_df)
except Exception:
    print("Per-seed metrics:")
    for r in sorted(seed_metric_rows, key=lambda x: (x["Seed"], x["Method"])):
        print(f"Seed={r['Seed']}  Method={r['Method']}")
        for k in metric_keys:
            v = r[k]
            print(f"  {k}: {v:.6g}" if np.isfinite(v) else f"  {k}: nan")

# -------- Mean/std summary --------
rows = []
for name, series in method_metric_series.items():
    row = {"Method": name}
    for k in metric_keys:
        mu, sig = mean_std_finite(series.get(k, []))
        row[f"{k}_mean"] = mu
        row[f"{k}_std"] = sig
    rows.append(row)

try:
    import pandas as pd
    cols = ["Method"] + [f"{k}_{s}" for k in metric_keys for s in ("mean", "std")]
    summary_df = pd.DataFrame(rows)[cols].sort_values("Method").reset_index(drop=True)
    print("Mean/std across seeds:")
    display(summary_df)
except Exception:
    print("Mean/std across seeds:")
    for r in rows:
        print("", r["Method"])
        for k in metric_keys:
            mu = r[f"{k}_mean"]
            sig = r[f"{k}_std"]
            print(f"  {k}: mean={mu:.6g}, std={sig:.6g}" if np.isfinite(mu) else f"  {k}: nan")


In [ ]:
# Re-display cached tables from the previous cell, filtered to selected metrics only.
selected_metrics = ["Err_arc", "CV", "GapRatio_m1", "IGD", "HV"]

try:
    per_seed_cols = ["Seed", "Method"] + selected_metrics
    print("Per-seed metrics (selected only):")
    display(per_seed_df[per_seed_cols])
except Exception:
    print("per_seed_df is not available. Run the previous metrics cell first.")

try:
    summary_cols = ["Method"] + [f"{k}_{s}" for k in selected_metrics for s in ("mean", "std")]
    print("Mean/std across seeds (selected only):")
    display(summary_df[summary_cols])
except Exception:
    print("summary_df is not available. Run the previous metrics cell first.")



## Notes

- This notebook uses **MO-Gymnasium `mo-mountaincar-v0`** with the default **3D** reward vector `[time, reverse, forward]`.
- Bi-objective mapping: `R1 = reward[0]` (time penalty), `R2 = reward[1]` (reverse penalty). The forward component is not used in the scalarization.
- Discrete deterministic actions: `0` = accelerate left, `1` = no acceleration, `2` = accelerate right.
- To reduce variance or strengthen policies, increase `TRAIN_EP_*` and/or `ALGO_SEEDS`.
